In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!git clone https://github.com/saimsvs/ML_OPS


In [ ]:
import os
os.chdir('/kaggle/working/ML_OPS')
print(os.listdir())

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    print(root)
    print(' dirs:', dirs)
    print(' files:', files[:5])
    

In [28]:
!ls /kaggle/working/ML_OPS


dataset_busi.py		ML_OPS		    README.md
evaluate_busi_lomix.py	preprocess_busi.py  train_busi.py
lomix.py		__pycache__	    unet_lomix.py


In [29]:
!cat /kaggle/working/ML_OPS/lomix.py
!cat /kaggle/working/ML_OPS/unet_lomix.py
!cat /kaggle/working/ML_OPS/train_busi.py
!cat /kaggle/working/ML_OPS/evaluate_busi_lomix.py


import logging
from itertools import combinations

import torch
import torch.nn as nn
import torch.nn.functional as F


class WeightedFusion(nn.Module):
    """
    Author's LoMix WeightedFusion implementation.

    Computes per-pixel attention weights for each prediction
    and fuses them using a weighted sum.
    """

    def __init__(self, num_stages, num_classes):
        super(WeightedFusion, self).__init__()

        self.weight_convs = nn.ModuleList(
            [
                nn.Conv2d(
                    num_classes,
                    1,
                    kernel_size=1
                )
                for _ in range(num_stages)
            ]
        )

    def forward(self, predictions):

        weight_maps = [
            conv(pred)
            for pred, conv in zip(
                predictions,
                self.weight_convs
            )
        ]

        weights = torch.stack(
            weight_maps,
            dim=1
        )

        weights = F.softmax

In [36]:
!python preprocess_busi.py

BUSI PREPROCESSING

Source: /kaggle/input/datasets/aryashah2k/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT
Output: /kaggle/working/ML_OPS/BUSI_processed

Finding BUSI images and masks...
Found 647 lesion images.

Class counts:
  Benign:    437
  Malignant: 210
  Total:     647

Images with multiple masks: 17

Creating 80/10/10 split...
Train: 517
Validation: 64
Test: 66

Processing train: 517 images
Finished train

Processing val: 64 images
Finished val

Processing test: 66 images
Finished test

DATASET VERIFICATION
train: 517 images, 517 masks
val  : 64 images, 64 masks
test : 66 images, 66 masks
------------------------------------------------------------
Total: 647 images
Verification passed!

PREPROCESSING COMPLETE

Processed dataset saved at:
/kaggle/working/ML_OPS/BUSI_processed


In [37]:
!python train_busi.py

Using device: cpu
Traceback (most recent call last):
  File "/kaggle/working/ML_OPS/train_busi.py", line 36, in <module>
    train_dataset = BUSIDataset(
                    ^^^^^^^^^^^^
  File "/kaggle/working/ML_OPS/dataset_busi.py", line 20, in __init__
    raise RuntimeError(
RuntimeError: No images found in /Users/saimasad/Desktop/ML_PROJ/BUSI_processed/train/images


In [39]:
from pathlib import Path

file = Path("/kaggle/working/ML_OPS/dataset_busi.py")

text = file.read_text()

text = text.replace(
    '/Users/saimasad/Desktop/ML_PROJ/BUSI_processed',
    '/kaggle/working/ML_OPS/BUSI_processed'
)

file.write_text(text)

print("dataset_busi.py updated successfully!")

dataset_busi.py updated successfully!


In [40]:
!python train_busi.py

Using device: cpu
Traceback (most recent call last):
  File "/kaggle/working/ML_OPS/train_busi.py", line 36, in <module>
    train_dataset = BUSIDataset(
                    ^^^^^^^^^^^^
  File "/kaggle/working/ML_OPS/dataset_busi.py", line 20, in __init__
    raise RuntimeError(
RuntimeError: No images found in /Users/saimasad/Desktop/ML_PROJ/BUSI_processed/train/images


In [41]:
!sed -n '1,55p' /kaggle/working/ML_OPS/train_busi.py


import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from unet_lomix import UNetLoMix
from dataset_busi import BUSIDataset
from lomix import CombinatorialMutationsLossModule


# ============================================================
# 1. DEVICE
# ============================================================

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)


# ============================================================
# 2. DATASET PATH
# ============================================================

DATASET_PATH = "/Users/saimasad/Desktop/ML_PROJ/BUSI_processed"


# ============================================================
# 3. DATASETS
# ============================================================

train_dataset = BUSIDataset(
    DATASET_PATH,
    split="train"
)

val_dataset = BUSIDataset(
    

In [43]:
from pathlib import Path

file = Path("/kaggle/working/ML_OPS/train_busi.py")

text = file.read_text()

text = text.replace(
    'DATASET_PATH = "/Users/saimasad/Desktop/ML_PROJ/BUSI_processed"',
    'DATASET_PATH = "/kaggle/working/ML_OPS/BUSI_processed"'
)

file.write_text(text)

print("train_busi.py updated successfully!")

train_busi.py updated successfully!


In [8]:
!git clone https://github.com/saimsvs/ML_OPS.git /kaggle/working/ML_OPS

Cloning into '/kaggle/working/ML_OPS'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 24 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 28.80 KiB | 9.60 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [9]:

!ls /kaggle/working

ML_OPS


In [20]:
!python train_busi.py

Using device: cuda
Traceback (most recent call last):
  File "/kaggle/working/ML_OPS/train_busi.py", line 36, in <module>
    train_dataset = BUSIDataset(
                    ^^^^^^^^^^^^
  File "/kaggle/working/ML_OPS/dataset_busi.py", line 20, in __init__
    raise RuntimeError(
RuntimeError: No images found in /kaggle/working/ML_OPS/BUSI_processed/train/images


In [15]:
%cd /kaggle/working/ML_OPS
!ls

/kaggle/working/ML_OPS
dataset_busi.py		lomix.py	    __pycache__  train_busi.py
evaluate_busi_lomix.py	preprocess_busi.py  README.md	 unet_lomix.py


In [19]:
from pathlib import Path

p = Path("train_busi.py")
text = p.read_text()

text = text.replace(
    '/Users/saimasad/Desktop/ML_PROJ/BUSI_processed',
    '/kaggle/working/ML_OPS/BUSI_processed'
)

p.write_text(text)

print("DATASET_PATH updated.")

DATASET_PATH updated.


In [26]:
!python train_busi.py


Using device: cuda
Traceback (most recent call last):
  File "/kaggle/working/ML_OPS/train_busi.py", line 36, in <module>
    train_dataset = BUSIDataset(
                    ^^^^^^^^^^^^
  File "/kaggle/working/ML_OPS/dataset_busi.py", line 20, in __init__
    raise RuntimeError(
RuntimeError: No images found in /kaggle/working/ML_OPS/BUSI_processed/train/images


In [25]:
from pathlib import Path

p = Path("preprocess_busi.py")
text = p.read_text()

text = text.replace(
    'SOURCE_DIR = Path("Dataset_BUSI_with_GT")',
    'SOURCE_DIR = Path("/kaggle/input/datasets/aryashah2k/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT")'
)

p.write_text(text)

print("SOURCE_DIR updated.")


SOURCE_DIR updated.


In [39]:
!python preprocess_busi.py

BUSI PREPROCESSING

Source: /kaggle/input/datasets/aryashah2k/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT
Output: /kaggle/working/ML_OPS/BUSI_processed

Finding BUSI images and masks...
Found 647 lesion images.

Class counts:
  Benign:    437
  Malignant: 210
  Total:     647

Images with multiple masks: 17

Creating 80/10/10 split...
Train: 517
Validation: 64
Test: 66

Processing train: 517 images
Finished train

Processing val: 64 images
Finished val

Processing test: 66 images
Finished test

DATASET VERIFICATION
train: 517 images, 517 masks
val  : 64 images, 64 masks
test : 66 images, 66 masks
------------------------------------------------------------
Total: 647 images
Verification passed!

PREPROCESSING COMPLETE

Processed dataset saved at:
/kaggle/working/ML_OPS/BUSI_processed


In [37]:
!python train_busi.py

Using device: cuda
train: 517 images found
val: 64 images found
train: 517 images found
val: 64 images found

Epoch 1/3
Batch 1/130 Loss: 27.4154
Batch 2/130 Loss: 21.1417
Batch 3/130 Loss: 29.4338
Batch 4/130 Loss: 28.1090
Batch 5/130 Loss: 25.7125
Batch 6/130 Loss: 23.2718
Batch 7/130 Loss: 27.9384
Batch 8/130 Loss: 26.6652
Batch 9/130 Loss: 26.1807
Batch 10/130 Loss: 27.2896
Batch 11/130 Loss: 24.2828
Batch 12/130 Loss: 27.4955
Batch 13/130 Loss: 26.5775
Batch 14/130 Loss: 27.2692
Batch 15/130 Loss: 25.8256
Batch 16/130 Loss: 23.9463
Batch 17/130 Loss: 23.5292
Batch 18/130 Loss: 26.2028
Batch 19/130 Loss: 22.5241
Batch 20/130 Loss: 25.9008
Batch 21/130 Loss: 21.8419
Batch 22/130 Loss: 24.5816
Batch 23/130 Loss: 27.0804
Batch 24/130 Loss: 26.4669
Batch 25/130 Loss: 24.2321
Batch 26/130 Loss: 24.5177
Batch 27/130 Loss: 26.4218
Batch 28/130 Loss: 27.0272
Batch 29/130 Loss: 24.9646
Batch 30/130 Loss: 27.4652
Batch 31/130 Loss: 27.1729
Batch 32/130 Loss: 25.7551
Batch 33/130 Loss: 24.756

In [38]:
from pathlib import Path

p = Path("/kaggle/working/ML_OPS/train_busi.py")
text = p.read_text()

text = text.replace("batch_size = 4", "batch_size = 16")
text = text.replace("num_epochs = 3", "num_epochs = 100")

p.write_text(text)

print("Updated batch size and epochs.")

Updated batch size and epochs.


In [43]:
%cd /kaggle/working/ML_OPS
!grep -nE "batch_size|num_epochs" train_busi.py

/kaggle/working/ML_OPS
48:    batch_size=16,
54:    batch_size=16,
174:num_epochs = 100
183:for epoch in range(num_epochs):
192:    print(f"Epoch {epoch + 1}/{num_epochs}")


In [42]:
from pathlib import Path

p = Path("/kaggle/working/ML_OPS/train_busi.py")
text = p.read_text()

text = text.replace("batch_size=4", "batch_size=16")


p.write_text(text)

print("Updated successfully.")

Updated successfully.


In [44]:
!python train_busi.py

Using device: cuda
train: 517 images found
val: 64 images found
train: 517 images found
val: 64 images found

Epoch 1/100
Batch 1/33 Loss: 25.9505
Batch 2/33 Loss: 27.9780
Batch 3/33 Loss: 26.3337
Batch 4/33 Loss: 25.8680
Batch 5/33 Loss: 25.2150
Batch 6/33 Loss: 26.9161
Batch 7/33 Loss: 24.0209
Batch 8/33 Loss: 23.5312
Batch 9/33 Loss: 25.8846
Batch 10/33 Loss: 24.8355
Batch 11/33 Loss: 25.0202
Batch 12/33 Loss: 26.3611
Batch 13/33 Loss: 24.9530
Batch 14/33 Loss: 24.7147
Batch 15/33 Loss: 26.0639
Batch 16/33 Loss: 25.7341
Batch 17/33 Loss: 25.5383
Batch 18/33 Loss: 26.0818
Batch 19/33 Loss: 25.0174
Batch 20/33 Loss: 23.8469
Batch 21/33 Loss: 24.9108
Batch 22/33 Loss: 24.2209
Batch 23/33 Loss: 25.9343
Batch 24/33 Loss: 24.8397
Batch 25/33 Loss: 25.2978
Batch 26/33 Loss: 25.7964
Batch 27/33 Loss: 24.5758
Batch 28/33 Loss: 24.0774
Batch 29/33 Loss: 25.0004
Batch 30/33 Loss: 25.5651
Batch 31/33 Loss: 25.2706
Batch 32/33 Loss: 24.4946
Batch 33/33 Loss: 22.1289

Epoch 1 average training los

In [46]:
import torch

path = "/kaggle/working/ML_OPS/busi_lomix_epoch_55.pth"

checkpoint = torch.load(path, map_location="cpu")

print(type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:")
    print(checkpoint.keys())

<class 'dict'>
Checkpoint keys:
dict_keys(['epoch', 'model_state_dict', 'lomix_state_dict', 'optimizer_state_dict', 'train_loss', 'val_loss'])


In [48]:
!sed -n '350,410p' /kaggle/working/ML_OPS/train_busi.py

        f"Epoch {epoch + 1} "
        f"validation loss: "
        f"{avg_val_loss:.4f}"
    )


    # ========================================================
    # 14. SAVE CHECKPOINT
    # ========================================================

    checkpoint = {
        "epoch": epoch + 1,

        "model_state_dict":
            model.state_dict(),

        "lomix_state_dict":
            lomix.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "train_loss":
            avg_train_loss,

        "val_loss":
            avg_val_loss
    }

    torch.save(
        checkpoint,
        f"busi_lomix_epoch_{epoch + 1}.pth"
    )

    print(
        f"Saved: "
        f"busi_lomix_epoch_{epoch + 1}.pth"
    )


    # ========================================================
    # 15. SAVE BEST MODEL
    # ========================================================

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        to

In [49]:
!grep -n "model =\|lomix =\|optimizer =\|best_val_loss\|num_epochs\|for epoch" /kaggle/working/ML_OPS/train_busi.py

66:model = UNetLoMix(
76:lomix = CombinatorialMutationsLossModule(
163:optimizer = torch.optim.Adam(
174:num_epochs = 100
176:best_val_loss = float("inf")
183:for epoch in range(num_epochs):
192:    print(f"Epoch {epoch + 1}/{num_epochs}")
394:    if avg_val_loss < best_val_loss:
396:        best_val_loss = avg_val_loss
417:    f"{best_val_loss:.4f}"


In [53]:
!python evaluate_busi_lomix.py


Using device: cuda
test: 66 images found
test: 66 images found
Loaded best model from epoch: 55
Training loss: 20.590260823567707
Validation loss: 21.112500190734863
Batch 1/17 Dice: 0.8973 IoU: 0.8160
Batch 2/17 Dice: 0.8709 IoU: 0.7781
Batch 3/17 Dice: 0.6368 IoU: 0.5675
Batch 4/17 Dice: 0.7398 IoU: 0.6374
Batch 5/17 Dice: 0.6612 IoU: 0.5795
Batch 6/17 Dice: 0.4355 IoU: 0.3578
Batch 7/17 Dice: 0.8894 IoU: 0.8039
Batch 8/17 Dice: 0.9313 IoU: 0.8740
Batch 9/17 Dice: 0.7451 IoU: 0.6772
Batch 10/17 Dice: 0.7578 IoU: 0.6499
Batch 11/17 Dice: 0.8857 IoU: 0.8000
Batch 12/17 Dice: 0.9142 IoU: 0.8454
Batch 13/17 Dice: 0.5006 IoU: 0.3991
Batch 14/17 Dice: 0.7970 IoU: 0.6713
Batch 15/17 Dice: 0.7289 IoU: 0.5999
Batch 16/17 Dice: 0.5625 IoU: 0.4533
Batch 17/17 Dice: 0.4053 IoU: 0.2912

TEST RESULTS
Test images: 66
Dice: 0.7368
IoU:  0.6458


In [51]:
!grep -n "DATASET_PATH\|CHECKPOINT" /kaggle/working/ML_OPS/evaluate_busi_lomix.py

27:DATASET_PATH = "/Users/saimasad/Desktop/ML_PROJ/BUSI_processed"
30:    DATASET_PATH,
54:# 4. LOAD BEST CHECKPOINT


In [54]:
!grep -n "dice\|iou\|total\|TEST RESULTS" /kaggle/working/ML_OPS/evaluate_busi_lomix.py

88:def calculate_dice(prediction, target, smooth=1e-5):
102:    dice = (
110:    return dice
117:def calculate_iou(prediction, target, smooth=1e-5):
137:    iou = (
143:    return iou
150:total_dice = 0.0
151:total_iou = 0.0
153:total_images = 0
210:        dice_scores = calculate_dice(
219:        iou_scores = calculate_iou(
230:        total_dice += dice_scores.sum().item()
231:        total_iou += iou_scores.sum().item()
233:        total_images += batch_size
237:            f"Dice: {dice_scores.mean().item():.4f} "
238:            f"IoU: {iou_scores.mean().item():.4f}"
246:average_dice = total_dice / total_images
247:average_iou = total_iou / total_images
252:print("TEST RESULTS")
256:    f"Test images: {total_images}"
260:    f"Dice: {average_dice:.4f}"
264:    f"IoU:  {average_iou:.4f}"
